In [ ]:
import json
import os
import platform
import shutil
import subprocess
from pathlib import Path

def check(args):
    if not shutil.which(args[0]):
        return {'available': False, 'reason': 'command missing'}
    try:
        run = subprocess.run(args, capture_output=True, text=True, timeout=12)
    except subprocess.TimeoutExpired:
        return {'available': False, 'reason': 'timeout'}
    return {'available': run.returncode == 0, 'output': run.stdout.strip() if run.returncode == 0 else 'command failed', 'exit': run.returncode}

checks = {
    'linux': platform.system(),
    'gpu': check(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader']),
    'docker_server': check(['docker', 'info', '--format', '{{.OSType}} {{.ServerVersion}}']),
    'docker_runtimes': check(['docker', 'info', '--format', '{{json .Runtimes}}']),
    'drive_mounted': Path('/content/drive/MyDrive').exists(),
}
checks['next'] = ('Return this inventory for image and toolkit verification' if checks['gpu']['available'] and checks['docker_server']['available'] else 'Docker GPU probe unavailable in this session')
print(json.dumps(checks, indent=2))
